# Hessian estimates from regression trees
### RA notes | Week 43, Oct 19–25, 2026

**What I'm trying to do.** Extend the first-derivative estimate in Nathan's *Regression Trees Know Calculus* paper to second derivatives. Nathan suggested comparing first-derivative estimates from neighboring parts of a sufficiently deep tree. For the first attempt, I'll use two levels of splits and the four grandchildren of one node.

My starting question was: **why should comparing the two child slopes give a Hessian entry, and what distance belongs in the denominator?** This notebook works through that question, checks the resulting formula on functions with known derivatives, and identifies what breaks with ordinary CART splits.

The second goal is sensitivity analysis: use the Hessian to see which *directions* have the most curvature, and check how stable the estimated Hessian is when the tree gets smaller or the data get noisier.

**References:** [Wycoff, Regression Trees Know Calculus, §3.1](https://arxiv.org/abs/2405.13846); meeting with Nathan on Oct 15, 2026; Nocedal and Wright, *Numerical Optimization*.

**Scope:** I'm not implementing physics-informed CART yet. The tree itself is a step function. These are attempts to recover derivatives of the smooth function behind the observations from the tree's stored averages.

## 1. Starting from Nathan's first-derivative estimate

Let \(f:\mathbb R^p\to\mathbb R\) be a smooth function and suppose the observations satisfy \(y_n=f(x_n)+\epsilon_n\). In Eq. (3) of the paper, for a node \(i\) splitting along coordinate \(\sigma_i\),

\[
\gamma_i=\frac{2(\hat\mu_r^i-\hat\mu_l^i)}
{u_{\sigma_i}^i-l_{\sigma_i}^i}.
\]

The numerator is the difference between the mean responses of its two children. The denominator is the width of the *parent* in the split coordinate, with the factor \(2\) accounting for the fact that in a midpoint split the child centers are half that width apart.

I'm first assuming balanced splits and uniform observations within the cells. I'll use exact cell averages for the basic algebra so I can separate approximation error from sampling noise. Later I'll drop the balanced-split assumption to see where the formula fails.

**One limitation worth keeping in mind:** having a very deep tree does not by itself make these derivatives accurate. Small cells need enough observations in them as well.

## 2. How to get \(H_{11}=\partial^2f/\partial x_1^2\)

Take a parent interval \([l,u]\) of width \(W=u-l\), with both its children splitting again along \(x_1\). For now every split is at the midpoint.

The four grandchildren, from left to right, are \(LL,LR,RL,RR\). Their widths are \(W/4\). Apply the original paper's formula within each child:

\[
\gamma_L=\frac{2(\hat\mu_{LR}-\hat\mu_{LL})}{W/2}
=\frac4W(\hat\mu_{LR}-\hat\mu_{LL}),
\]

\[
\gamma_R=\frac{2(\hat\mu_{RR}-\hat\mu_{RL})}{W/2}
=\frac4W(\hat\mu_{RR}-\hat\mu_{RL}).
\]

### Why is the *second* denominator \(W/2\), not \(W/4\)?

I was mixing up the width of a **grandchild** with the distance between the two **slope estimates**. Those are different.

- \(\gamma_L\) is a slope estimate for the *left child*. With balanced splits, I can treat it as an approximation to \(f'\) near the left child's center \(c_L=l+W/4\).
- \(\gamma_R\) is a slope estimate for the *right child*, near \(c_R=l+3W/4\).
- The distance between those two locations is

\[
c_R-c_L=(l+3W/4)-(l+W/4)=W/2.
\]

For example, on \([0,1]\), the grandchildren have width \(1/4\), but the left and right child centers are \(1/4\) and \(3/4\). The two slopes are being compared across a distance of \(1/2\).

Since the second derivative is the rate of change of the **first derivative**, my proposed finite difference is

\[
\widetilde H_{11}
=\frac{\gamma_R-\gamma_L}{c_R-c_L}
=\frac{\gamma_R-\gamma_L}{W/2}.
\]

Now plug in both estimates:

\[
\begin{aligned}
\widetilde H_{11}
&=\frac{\frac4W(\hat\mu_{RR}-\hat\mu_{RL})-
             \frac4W(\hat\mu_{LR}-\hat\mu_{LL})}{W/2}\\
&=\boxed{\frac8{W^2}
(\hat\mu_{RR}-\hat\mu_{RL}-\hat\mu_{LR}+\hat\mu_{LL})}.
\end{aligned}\tag{A}
\]

**What I can claim so far:** Eq. (A) is a candidate diagonal-Hessian estimator for this balanced two-level geometry. It is not an established formula for arbitrary CART trees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from numpy.polynomial.legendre import leggauss
BLUE, GOLD, RED = "#2874a6", "#bd7f33", "#bb4c44"

def diagram_diagonal():
    fig,(tree,intervals)=plt.subplots(2,1,figsize=(9,5),
                                      gridspec_kw={"height_ratios":[1.3,1]})
    pos={"P":(2,2),"L":(1,1),"R":(3,1),
         "LL":(.5,0),"LR":(1.5,0),"RL":(2.5,0),"RR":(3.5,0)}
    for parent,kid in [("P","L"),("P","R"),("L","LL"),("L","LR"),
                       ("R","RL"),("R","RR")]:
        tree.plot([pos[parent][0],pos[kid][0]],
                  [pos[parent][1]-.12,pos[kid][1]+.12],color=".5",lw=1.5)
    for name,(x,y) in pos.items():
        color=RED if name=="P" else (BLUE if name.startswith("L") else GOLD)
        tree.add_patch(Rectangle((x-.3,y-.15),.6,.3,facecolor=color,
                                 edgecolor=color,alpha=.25))
        tree.text(x,y,name,ha="center",va="center",fontsize=12)
    tree.set(xlim=(0,4),ylim=(-.25,2.3),title="Tree topology: parent, children, grandchildren")
    tree.axis("off")
    for j,name in enumerate(["LL","LR","RL","RR"]):
        intervals.add_patch(Rectangle((j/4,.2),.25,.56,facecolor=BLUE if j<2 else GOLD,
                                      edgecolor="white",alpha=.28))
        intervals.text((j+.5)/4,.47,name,ha="center",va="center",fontsize=12)
    intervals.annotate("",xy=(.75,.9),xytext=(.25,.9),
                       arrowprops={"arrowstyle":"<->"})
    intervals.text(.5,.94,"distance between child centers = W/2",ha="center",fontsize=10)
    intervals.set(xlim=(0,1),ylim=(0,1.12),xticks=[0,.25,.5,.75,1],
                  xlabel="x1 (parent is [0,1])",yticks=[])
    intervals.spines[["top","left","right"]].set_visible(False)
    fig.tight_layout();plt.show()

diagram_diagonal()

## 3. Check the calculation against \(f(x)=x^2\)

Rather than use \(f\) evaluated at the midpoint of each cell, I'll use the **average response over that cell**, since those are the quantities the tree is storing. With a uniform input distribution,

\[
\mu_{[a,b]}=\frac1{b-a}\int_a^b f(x)\,dx.
\]

For \(f(x)=x^2\) on \([0,1]\), the exact means from left to right are

\[
(\mu_{LL},\mu_{LR},\mu_{RL},\mu_{RR})=(1,7,19,37)/48.
\]

Plugging them into Eq. (A):

\[
\widetilde H_{11}
=\frac{8}{1^2}\left(\frac{37-19-7+1}{48}\right)=2.
\]

That matches \(f''(x)=2\). This is a useful arithmetic check, but one quadratic working is not a proof of consistency.

In [ ]:
def mean_1d(f,a,b,q=32):
    z,w=leggauss(q)
    x=(a+b)/2+(b-a)*z/2
    return float(np.dot(w,f(x))/2)

def four_balanced_means(f,center=.5,width=1.):
    e=np.linspace(center-width/2,center+width/2,5)
    return np.array([mean_1d(f,e[i],e[i+1]) for i in range(4)])

def balanced_h11(means,width=1.):
    LL,LR,RL,RR=means
    return 8*(RR-RL-LR+LL)/width**2

quadratic_means=four_balanced_means(lambda x:x**2)
print("Exact x^2 means:",quadratic_means)
print("H11 estimate:",balanced_h11(quadratic_means),"true:",2)
assert np.allclose(quadratic_means,np.array([1,7,19,37])/48)
assert np.isclose(balanced_h11(quadratic_means),2,atol=1e-12)

### Check the bias on a function with changing curvature

For \(f(x)=x^4\), with the parent centered at \(c=1/2\) and \(W=1\), the true value is \(f''(c)=3\). The tree formula gives \(27/8=3.375\).

A Taylor expansion for the **balanced, uniformly averaged** stencil gives

\[
\widetilde H_{11}(c)
=f''(c)+\frac{W^2}{64}f^{(4)}(c)+O(W^4).
\tag{B}
\]

Here \(f^{(4)}(c)=24\), so the predicted bias is \(24/64=3/8\), exactly the difference I found.

The point of the next plot is to check the \(W^2\) behavior numerically on \(e^x\). This expansion assumes enough smoothness for the displayed remainder and isn't a convergence theorem for CART.

In [ ]:
fourth_means=four_balanced_means(lambda x:x**4)
fourth_est=balanced_h11(fourth_means)
print("x^4 estimate:",fourth_est,"true:",3,"bias:",fourth_est-3)
assert np.isclose(fourth_est,27/8,atol=1e-12)
assert np.isclose(fourth_est-3,24/64,atol=1e-12)

widths=np.array([.9,.7,.5,.35,.25,.18,.12])
errors=np.array([balanced_h11(four_balanced_means(np.exp,.5,w),w)-np.exp(.5)
                 for w in widths])
fig,ax=plt.subplots(figsize=(7,3.7))
ax.loglog(widths,np.abs(errors),"o-",label="Observed |error| for exp(x)")
ax.loglog(widths,np.exp(.5)*widths**2/64,"--",
          label="Predicted leading bias magnitude")
ax.set(xlabel="Parent width W",ylabel="Absolute error",
       title="Balanced geometry: second-derivative bias is order W²")
ax.legend();ax.grid(True,which="both",alpha=.25);plt.show()

## 4. What happens if CART does not split at the midpoint?

CART chooses a threshold from the data, so the four grandchildren usually won't be equal-width. Let the parent be \([l,u]\), split at \(t\); the left child splits at \(s_L\), and the right at \(s_R\), where

\[
l<s_L<t<s_R<u.
\]

Put \(W_L=t-l,\ W_R=u-t,\ W=u-l\). The paper's first-order rule still gives

\[
\gamma_L=\frac{2(\hat\mu_{LR}-\hat\mu_{LL})}{W_L},
\qquad
\gamma_R=\frac{2(\hat\mu_{RR}-\hat\mu_{RL})}{W_R}.
\]

If I simply divide their difference by the child-center distance \(W/2\), I get

\[
\widetilde H_{11}^{\rm naive}
=\frac4W\left[
\frac{\hat\mu_{RR}-\hat\mu_{RL}}{W_R}
-\frac{\hat\mu_{LR}-\hat\mu_{LL}}{W_L}
\right].
\tag{C}
\]

**Problem:** when the inner split is uneven, the estimated first derivative might not be located at the *child center*. I may be dividing the difference of slopes by the wrong spatial separation.

I'll test this using a quadratic, because if the method misses a constant second derivative, that problem won't disappear just by trying a more complicated function.

In [ ]:
def unequal_h11(f,edges):
    l,sL,t,sR,u=map(float,edges)
    assert l<sL<t<sR<u
    means=np.array([mean_1d(f,a,b) for a,b in zip(edges[:-1],edges[1:])])
    LL,LR,RL,RR=means
    gamma_L=2*(LR-LL)/(t-l)
    gamma_R=2*(RR-RL)/(u-t)
    return (gamma_R-gamma_L)/((u-l)/2),means,(gamma_L,gamma_R)

edges=[0,1/8,1/2,7/8,1]
unequal,means,gradients=unequal_h11(lambda x:x**2,edges)
print("Edges:",edges,"means:",means,"gradients:",gradients)
print("Naive H11:",unequal,"true H11:",2,"absolute error:",abs(unequal-2))
assert np.isclose(unequal,7/3,atol=1e-12)

fig,ax=plt.subplots(figsize=(8,2.2))
for j,(a,b) in enumerate(zip(edges[:-1],edges[1:])):
    ax.add_patch(Rectangle((a,.2),b-a,.6,facecolor=BLUE if j<2 else GOLD,
                           edgecolor="black",alpha=.22))
    ax.text((a+b)/2,.5,["LL","LR","RL","RR"][j],ha="center",va="center")
ax.set(xlim=(-.02,1.02),ylim=(0,1),yticks=[],xticks=edges,
       title="Unequal thresholds change effective derivative locations")
plt.show()

### An alternative worth comparing, not a replacement I've justified yet

For the uneven example, the naive formula gives \(7/3\) instead of the true value \(2\). Even shrinking the parent while keeping the same proportions leaves this discrepancy for \(x^2\).

One idea is to account for the geometry explicitly: fit a local quadratic whose **cell averages** match the observed grandchild means, using the known averages of \(x\) and \(x^2\) on each interval. Twice the fitted quadratic coefficient estimates \(f''\).

This isn't the TBGE from the paper. I'm including it to test whether the failure is caused by the particular normalization or by the limited information available in four cells.

In [ ]:
def quadratic_moment_h11(f,edges):
    center=(edges[0]+edges[-1])/2
    A=[];y=[]
    for a,b in zip(edges[:-1],edges[1:]):
        ex=(a+b)/2
        ex2=(a*a+a*b+b*b)/3
        A.append([1,ex-center,ex2-2*center*ex+center**2])
        y.append(mean_1d(f,a,b))
    beta=np.linalg.lstsq(np.asarray(A),np.asarray(y),rcond=None)[0]
    return 2*beta[2]

corrected=quadratic_moment_h11(lambda x:x**2,edges)
print("Moment-based quadratic H11:",corrected)
assert np.isclose(corrected,2,atol=1e-12)

## 5. Mixed partials: \(H_{12}=\partial^2 f/\partial x_1\partial x_2\)

For the off-diagonal Hessian entry I need a different arrangement. The parent first splits along \(x_1\). Then **both** children split along \(x_2\). In the balanced, aligned case the four grandchildren are \(LD,LU,RD,RU\): left/right along \(x_1\), down/up along \(x_2\).

The slopes *in the \(x_2\) direction* are

\[
\gamma_{L,2}=\frac{2(\hat\mu_{LU}-\hat\mu_{LD})}{W_2},
\qquad
\gamma_{R,2}=\frac{2(\hat\mu_{RU}-\hat\mu_{RD})}{W_2}.
\]

They are located about \(W_1/2\) apart along \(x_1\). So the second difference, **in the \(x_1\) direction**, is

\[
\begin{aligned}
\widetilde H_{12}
&=\frac{\gamma_{R,2}-\gamma_{L,2}}{W_1/2}\\
&=\boxed{\frac4{W_1W_2}
(\hat\mu_{RU}-\hat\mu_{RD}-\hat\mu_{LU}+\hat\mu_{LD})}.
\end{aligned}\tag{D}
\]

This doesn't estimate all of \(H\) from four cells. For \(H_{11}\) I'd need repeated splits along \(x_1\); for \(H_{22}\), repeated splits along \(x_2\). The mixed partial instead uses one of each.

In [ ]:
def diagram_mixed():
    fig,ax=plt.subplots(figsize=(6,5))
    for ix in (0,1):
        for iy in (0,1):
            lab=("L" if ix==0 else "R")+("D" if iy==0 else "U")
            ax.add_patch(Rectangle((ix/2,iy/2),.5,.5,
                 facecolor=BLUE if ix==0 else GOLD,edgecolor="white",alpha=.23))
            ax.text(ix/2+.25,iy/2+.25,lab,ha="center",va="center",fontsize=15)
    ax.axvline(.5,color=RED,lw=3,label="First split: x1")
    ax.axhline(.5,color=BLUE,lw=2,ls="--",label="Second-level splits: x2")
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="x1",ylabel="x2",
           title="Four grandchildren for a mixed partial")
    ax.set_aspect("equal");ax.legend(loc="upper center",bbox_to_anchor=(.5,-.14))
    fig.tight_layout();plt.show()

diagram_mixed()

In [ ]:
def mean_2d(f,x0,x1,y0,y1,q=16):
    z,w=leggauss(q)
    X=(x0+x1)/2+(x1-x0)*z/2
    Y=(y0+y1)/2+(y1-y0)*z/2
    return float(np.sum(f(X[:,None],Y[None,:])*w[:,None]*w[None,:])/4)

def mixed_quadrant_means(f):
    return { "LD":mean_2d(f,0,.5,0,.5),
             "LU":mean_2d(f,0,.5,.5,1),
             "RD":mean_2d(f,.5,1,0,.5),
             "RU":mean_2d(f,.5,1,.5,1)}

def balanced_h12(means,W1=1,W2=1):
    return 4*(means["RU"]-means["RD"]-means["LU"]+means["LD"])/(W1*W2)

f=lambda x1,x2:x1**2+3*x1*x2+2*x2**2
quad=mixed_quadrant_means(f)
print("Quadrant means:",quad)
print("Estimated H12:",balanced_h12(quad),"true H12:",3)
assert np.allclose([quad[k] for k in ["LD","LU","RD","RU"]],
                   [7/16,29/16,21/16,55/16],atol=1e-12)
assert np.isclose(balanced_h12(quad),3,atol=1e-12)

### Verify the mixed derivative on a quadratic surface

Take

\[
f(x_1,x_2)=x_1^2+3x_1x_2+2x_2^2.
\]

Its first derivatives are

\[
\frac{\partial f}{\partial x_1}=2x_1+3x_2,\qquad
\frac{\partial f}{\partial x_2}=3x_1+4x_2.
\]

Now differentiate each once more:

\[
H=\nabla^2 f=
\begin{pmatrix}
2&3\\
3&4
\end{pmatrix}.
\]

The balanced quadrant means (LD, LU, RD, RU) are \((7,29,21,55)/16\). Equation (D) gives

\[
\widetilde H_{12}=4(55-21-29+7)/16=3.
\]

So the off-diagonal stencil recovers the mixed derivative here. This is the same idea as the diagonal case: **take a derivative estimate in one direction, then measure how that estimate changes in another direction.**

## 6. How sensitive is the Hessian estimate to noise?

For the balanced one-dimensional stencil, suppose each of the four estimated cell means has independent measurement error with variance \(\sigma^2/n\), where \(n\) is the number of observations **per** grandchild.

The coefficient on each mean in Eq. (A) has magnitude \(8/W^2\). Adding the four independent variances gives

\[
\operatorname{Var}(\widetilde H_{11})
=4\left(\frac8{W^2}\right)^2\frac{\sigma^2}{n}
=\frac{256\sigma^2}{nW^4},
\]

\[
\operatorname{SD}(\widetilde H_{11})
=\frac{16\sigma}{W^2\sqrt n}.
\tag{E}
\]

So smaller cells help with approximation bias but make the estimator more sensitive to noisy means. In this simple fixed-partition setup, I need \(nW^4\to\infty\) to control the variance. With roughly \(n\propto NW\) in 1D, the condition becomes \(NW^5\to\infty\).

This ignores the randomness from CART selecting its splits, so I shouldn't use it as a theoretical rate for the actual algorithm.

In [ ]:
rng=np.random.default_rng(20261021)
sigma,n,W,reps=.35,400,.6,20000
e=rng.normal(0,sigma/np.sqrt(n),size=(reps,4))
noise=8*(e[:,3]-e[:,2]-e[:,1]+e[:,0])/W**2
predicted_sd=16*sigma/(W**2*np.sqrt(n))
print("Observed SD:",noise.std(ddof=1),"predicted SD:",predicted_sd)
assert abs(noise.std(ddof=1)/predicted_sd-1)<.04
fig,ax=plt.subplots(figsize=(7,3))
ax.hist(noise,bins=55,density=True,alpha=.7)
ax.axvline(predicted_sd,color=RED,ls="--",label="+1 theoretical SD")
ax.axvline(-predicted_sd,color=RED,ls="--",label="-1 theoretical SD")
ax.set(xlabel="Hessian estimation noise",ylabel="Density",
       title="Noise amplification from differencing four sample means")
ax.legend();plt.show()

## 7. Sensitivity analysis: what does the Hessian tell me?

First I want to separate **sensitivity of the estimated Hessian** (Section 6) from **sensitivity of the underlying function**.

The gradient tells me how fast \(f\) changes at a point. The Hessian tells me how the gradient itself changes, so it captures local **curvature**. For a unit direction \(v\), the second directional derivative is

\[
D_v^2 f(x)=v^\top H(x)v.
\]

The eigenvectors of the symmetric Hessian identify orthogonal curvature directions. The eigenvalues tell me the signed curvature along those directions. Large **absolute** eigenvalues mean large curvature; positive and negative signs indicate whether the function curves up or down.

For the quadratic above, \(H\) is constant:

\[
H=\begin{pmatrix}2&3\\3&4\end{pmatrix},
\qquad \det(H)=8-9=-1.
\]

Since the determinant is negative, \(H\) has one positive and one negative eigenvalue. This is a saddle, not a convex bowl. The eigenvalues are \(3\pm\sqrt{10}\).

I'll compute the eigenvectors and compare \(v^\top H v\) across directions in the next cell. **This is local Hessian eigenanalysis.** Nathan mentioned *principal Hessian directions* as a later sensitivity-analysis method; I still need to check its precise population definition before calling this example a full PHD analysis.

In [ ]:
# Local curvature sensitivity for the known quadratic.
H=np.array([[2.,3.],[3.,4.]])
lam,V=np.linalg.eigh(H)   # eigenvalues in ascending order; eigenvectors are columns
print("Analytic Hessian:\n",H)
print("Eigenvalues:",lam)
print("Eigenvectors (columns):\n",V)
print("Check H v = lambda v:",np.allclose(H@V,V*lam))
assert np.allclose(lam,[3-np.sqrt(10),3+np.sqrt(10)])

theta=np.linspace(0,2*np.pi,500)
directions=np.stack([np.cos(theta),np.sin(theta)],axis=1)
curvature=np.einsum("ni,ij,nj->n",directions,H,directions)
fig,ax=plt.subplots(figsize=(7.5,3.6))
ax.plot(theta,curvature,label=r"$v(\theta)^T H v(\theta)$")
ax.axhline(lam[0],linestyle="--",alpha=.7,label="Smallest eigenvalue")
ax.axhline(lam[-1],linestyle="--",alpha=.7,label="Largest eigenvalue")
ax.axhline(0,color="black",lw=.8)
ax.set(xlabel="Direction angle (radians)",ylabel="Second directional derivative",
       title="Curvature depends on direction")
ax.legend();ax.grid(alpha=.2);plt.show()
print("Greatest magnitude curvature:",np.max(abs(lam)),
      "along eigenvector",V[:,np.argmax(abs(lam))])

### Sensitivity of the *estimate*, not just the function

For the balanced \(H_{11}\) stencil, the standard deviation of the error behaves like \(16\sigma/(W^2\sqrt n)\). I'll plot that as a function of \(W\) and \(n\).

This is useful because my Hessian eigenvectors will be unstable if the estimated matrix entries are too noisy, especially when the true eigenvalues are close to each other. I won't claim a general eigenvector accuracy theorem from this plot. It's a basic diagnostic for how much data a small cell needs.

In [ ]:
# Theoretical measurement-noise sensitivity for the fixed four-cell stencil.
width_grid=np.array([.2,.3,.45,.65,.9])
samples_grid=np.array([25,100,400,1600])
sigma=.1
fig,ax=plt.subplots(figsize=(7.5,3.7))
for n_cell in samples_grid:
    sd=16*sigma/(width_grid**2*np.sqrt(n_cell))
    ax.plot(width_grid,sd,"o-",label=f"{n_cell} samples / grandchild")
ax.set(xlabel="Parent width W",ylabel="Theoretical SD of H11 estimate",
       title="Smaller cells amplify noise; more observations reduce it")
ax.legend();ax.grid(alpha=.2);plt.show()

## 8. Status and next step

**What I worked out in these notes**
- Reproduced Nathan's first-derivative formula and substituted it twice to propose a second-derivative estimator.
- Checked why the outer denominator is \(W/2\): it's the separation between the two estimated *slopes*, not the width of the grandchildren.
- Checked the diagonal formula on \(x^2\) and \(x^4\), and wrote down the leading bias in the balanced case.
- Derived a mixed-partial formula for two different split directions and checked it on a known quadratic Hessian.
- Found a counterexample for unequal splits, and compared the naive stencil with a local quadratic reconstruction.
- Set up an initial noise-sensitivity check and a direction-of-curvature example below.

**What I haven't figured out**
- A general second-order estimator that accounts correctly for arbitrary CART thresholds and uneven input density.
- Whether requiring certain repeated splits is the best way to make enough derivative information available.
- How to estimate a complete, stable Hessian at each location efficiently from an actual fitted tree.
- How any of this transfers to a power-grid example.

**Next implementation task:** Extract a parent, its two children, its four grandchildren, their split features, bounds, and means from a fitted tree. Calculate the appropriate Hessian entry only when the necessary split pattern is present, and compare it against an analytic derivative. I should *not* fill in missing Hessian entries with zeros.